# 🎓 Predicción del Rendimiento Académico de Estudiantes
## Proyecto de Machine Learning con PyCaret (Regresión)
### Trabajo de Maestría

---

**Autor:** _(completar con tu nombre)_
**Maestría:** _(completar)_
**Fecha:** _(completar)_

---

## 📌 Objetivo del proyecto

Construir, comparar, **validar** y optimizar modelos de **regresión** que permitan **predecir el puntaje de matemáticas** (`math score`) de un estudiante a partir de variables demográficas y académicas (género, grupo étnico, nivel educativo de los padres, tipo de almuerzo, y si tomó o no un curso de preparación para exámenes), utilizando la librería **PyCaret**.

## 📌 Justificación

Predecir el rendimiento académico permite a instituciones educativas:
- Identificar de forma temprana a estudiantes en riesgo de bajo desempeño.
- Diseñar intervenciones focalizadas (tutorías, cursos de preparación, apoyo socioeconómico).
- Entender qué variables socio-demográficas se asocian más fuertemente con el desempeño académico.

Este tipo de análisis es ampliamente utilizado en **Learning Analytics** y **Educational Data Mining**, áreas de investigación activa dentro de la Inteligencia Artificial aplicada a la educación.

## 📌 Dataset

**Nombre:** Students Performance in Exams
**Fuente:** Kaggle — https://www.kaggle.com/datasets/spscientist/students-performance-in-exams
**Autor original:** Royce Kimmons (datos sintéticos con fines educativos)
**Tamaño:** 1000 filas × 8 columnas

| Variable | Tipo | Descripción |
|---|---|---|
| gender | Categórica | Género del estudiante (male/female) |
| race/ethnicity | Categórica | Grupo étnico (group A-E, anonimizado) |
| parental level of education | Categórica | Nivel educativo máximo alcanzado por los padres |
| lunch | Categórica | Tipo de almuerzo (standard / free-reduced) — proxy de nivel socioeconómico |
| test preparation course | Categórica | Si completó (`completed`) o no (`none`) un curso de preparación |
| math score | Numérica | Puntaje en matemáticas (0-100) — **variable objetivo (target)** |
| reading score | Numérica | Puntaje en lectura (0-100) |
| writing score | Numérica | Puntaje en escritura (0-100) |

## 📌 Estructura de este notebook

0. **Fundamentos teóricos**: cómo funciona matemáticamente la regresión, con ejemplo ilustrado paso a paso
1. Instalación de librerías
2. Configuración de la API de Kaggle
3. Descarga del dataset
4. Carga de datos y **validación del esquema** (calidad de datos)
5. Análisis Exploratorio de Datos (EDA) con múltiples gráficas
6. Configuración del experimento en PyCaret
7. Comparación automática de modelos
8. Optimización de hiperparámetros (tuning)
9. **Validación del modelo**: estabilidad entre folds, curva de aprendizaje, curva de validación
10. Evaluación visual (residuos, error, importancia de variables, parámetros, árbol, manifold)
11. Interpretabilidad con SHAP
12. Ensamblado de modelos (blending / stacking)
13. **Validación final sobre el conjunto de prueba** (hold-out) con pruebas estadísticas
14. Finalización y guardado del modelo
15. **Ejemplos de uso**: predicción individual, por lote (batch) y carga del modelo persistido
16. Conclusiones y recomendaciones para la tesis

---


## 0️⃣ Fundamentos teóricos: ¿cómo funciona la regresión?

Antes de implementar el proyecto, es fundamental entender **qué hace matemáticamente un modelo de regresión** y cómo aprende. Esta sección está pensada para incluirse (resumida) en el capítulo de **Marco Teórico** de la tesis.

### 0.1 ¿Qué es un problema de regresión?

La regresión es una tarea de **aprendizaje supervisado** cuyo objetivo es aprender una función $f$ que relacione un conjunto de variables predictoras (features) $X = (x_1, x_2, ..., x_n)$ con una variable objetivo **continua** $y$:

$$y = f(X) + \varepsilon$$

Donde $\varepsilon$ representa el error irreducible (ruido) que ningún modelo puede explicar. El objetivo del algoritmo de aprendizaje es encontrar una función $\hat{f}$ que **minimice el error** entre el valor real $y$ y el valor predicho $\hat{y} = \hat{f}(X)$.

Esto la diferencia de la **clasificación**, donde la variable objetivo es categórica (ej. "aprobado/reprobado"). En nuestro caso, `math score` es un valor numérico continuo entre 0 y 100, por lo que es un problema de **regresión**.

### 0.2 Regresión Lineal — el caso más simple

El modelo más simple e interpretable es la **regresión lineal múltiple**, que asume una relación lineal entre las variables predictoras y el objetivo:

$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + ... + \beta_n x_n$$

- $\beta_0$: intercepto (valor predicho cuando todas las variables son 0).
- $\beta_1, ..., \beta_n$: coeficientes que indican **cuánto cambia** $\hat{y}$ por cada unidad de cambio en $x_i$, manteniendo las demás constantes.

**¿Cómo se "aprenden" los coeficientes?** Minimizando una función de costo, típicamente el **Error Cuadrático Medio (MSE)**:

$$MSE = \frac{1}{m}\sum_{i=1}^{m}(y_i - \hat{y}_i)^2$$

Esto se resuelve de forma cerrada (mínimos cuadrados ordinarios) o iterativamente mediante **descenso de gradiente**, actualizando los coeficientes en la dirección que reduce el error:

$$\beta_j := \beta_j - \alpha \frac{\partial MSE}{\partial \beta_j}$$

donde $\alpha$ es la tasa de aprendizaje (*learning rate*).

### 0.3 Modelos basados en árboles (Random Forest, XGBoost, LightGBM, etc.)

Los modelos que suelen ganar la comparación en `compare_models()` (Random Forest, Gradient Boosting, XGBoost, LightGBM, CatBoost) **no asumen una relación lineal**. Funcionan así:

1. Un **árbol de decisión de regresión** divide repetidamente el espacio de variables en regiones, buscando en cada división (split) el punto de corte que **minimiza la varianza** (o el MSE) de la variable objetivo dentro de cada región resultante.
2. Cada hoja del árbol final predice el **promedio** de los valores de entrenamiento que cayeron en esa región.
3. **Random Forest / Extra Trees (Bagging):** se entrenan muchos árboles sobre muestras aleatorias de los datos (bootstrap) y se promedian sus predicciones, reduciendo la varianza y el sobreajuste.
4. **Gradient Boosting / XGBoost / LightGBM / CatBoost (Boosting):** se entrenan árboles de forma secuencial, donde cada nuevo árbol corrige los errores (residuos) que dejaron los árboles anteriores, sumando sus predicciones ponderadas:

$$\hat{y} = \sum_{k=1}^{K} \eta \cdot h_k(X)$$

donde $h_k$ es el árbol número $k$ y $\eta$ es la tasa de aprendizaje del boosting.

### 0.4 Métricas de evaluación (qué significan realmente)

| Métrica | Fórmula | Interpretación |
|---|---|---|
| **MAE** (Error Absoluto Medio) | $\frac{1}{m}\sum \lvert y_i - \hat{y}_i \rvert$ | Error promedio en las mismas unidades que $y$ (puntos de puntaje). Poco sensible a outliers. |
| **MSE** (Error Cuadrático Medio) | $\frac{1}{m}\sum (y_i - \hat{y}_i)^2$ | Penaliza más los errores grandes (por el cuadrado). Unidades al cuadrado, difícil de interpretar directamente. |
| **RMSE** (Raíz del MSE) | $\sqrt{MSE}$ | Igual que MSE pero en las unidades originales de $y$. Es la métrica principal que usamos para ordenar modelos. |
| **R²** (Coeficiente de determinación) | $1 - \frac{\sum(y_i-\hat{y}_i)^2}{\sum(y_i-\bar{y})^2}$ | Proporción de la varianza de $y$ explicada por el modelo. Va de 0 a 1 (más cercano a 1 = mejor). |
| **MAPE** | $\frac{1}{m}\sum \left\lvert \frac{y_i-\hat{y}_i}{y_i} \right\rvert$ | Error porcentual promedio, útil para comparar en términos relativos. |
| **RMSLE** | $\sqrt{\frac{1}{m}\sum(\log(1+y_i)-\log(1+\hat{y}_i))^2}$ | Similar al RMSE pero en escala logarítmica; penaliza menos los errores en valores grandes. |

### 0.5 ¿Por qué usamos validación cruzada (Cross-Validation)?

Si evaluáramos el modelo solo con una única partición train/test, el resultado podría depender de la "suerte" de esa partición particular. La **validación cruzada de k folds (k-fold CV)** soluciona esto:

1. Los datos de entrenamiento se dividen en $k$ partes (folds) de tamaño similar (en nuestro caso $k=10$).
2. El modelo se entrena $k$ veces, usando en cada iteración $k-1$ folds para entrenar y el fold restante para validar.
3. Se calcula la métrica (ej. RMSE) en cada una de las $k$ iteraciones y se reporta el **promedio** y la **desviación estándar**.

Esto es exactamente lo que PyCaret hace automáticamente dentro de `compare_models()`, `tune_model()` y `create_model()`, y es la razón por la que más adelante analizamos la **estabilidad entre folds** como parte de la validación del modelo.

### 0.6 Ejemplo ilustrativo con datos sintéticos (antes de usar el dataset real)

Para que el concepto quede completamente claro, ajustamos "a mano" una regresión lineal simple sobre datos sintéticos, visualizando cómo la línea se ajusta a los puntos y cómo se ven los residuos. Esto sirve como puente conceptual antes de aplicar los mismos principios (pero con algoritmos mucho más sofisticados) al dataset real de estudiantes.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Generamos datos sintéticos: y = 3x + 5 + ruido
np.random.seed(42)
x_sint = np.linspace(0, 10, 60)
y_sint = 3 * x_sint + 5 + np.random.normal(0, 3, size=len(x_sint))

# Ajuste manual por mínimos cuadrados (fórmula cerrada): beta = (X^T X)^-1 X^T y
X_matriz = np.column_stack([np.ones_like(x_sint), x_sint])
beta = np.linalg.inv(X_matriz.T @ X_matriz) @ X_matriz.T @ y_sint
intercepto, pendiente = beta
y_pred_sint = intercepto + pendiente * x_sint

print(f"Ecuación aprendida: y = {pendiente:.2f}·x + {intercepto:.2f}")
print(f"(Valores reales usados para generar los datos: pendiente=3, intercepto=5)")

# --- Gráfico A: Ajuste de la recta de regresión ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(x_sint, y_sint, alpha=0.6, color='steelblue', label='Datos observados')
axes[0].plot(x_sint, y_pred_sint, color='red', linewidth=2, label=f'Recta ajustada: y={pendiente:.2f}x+{intercepto:.2f}')
for xi, yi, ypi in zip(x_sint, y_sint, y_pred_sint):
    axes[0].plot([xi, xi], [yi, ypi], color='gray', alpha=0.3, linewidth=1)  # líneas de residuo
axes[0].set_title('Regresión lineal simple: ajuste y residuos (líneas grises)')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].legend()

# --- Gráfico B: Función de costo (MSE) en función de la pendiente ---
pendientes_prueba = np.linspace(0, 6, 100)
mse_por_pendiente = [np.mean((y_sint - (intercepto + m*x_sint))**2) for m in pendientes_prueba]

axes[1].plot(pendientes_prueba, mse_por_pendiente, color='darkorange')
axes[1].axvline(pendiente, color='red', linestyle='--', label=f'Mínimo en pendiente={pendiente:.2f}')
axes[1].set_title('Función de costo (MSE) según el valor de la pendiente')
axes[1].set_xlabel('Valor de la pendiente (β₁)')
axes[1].set_ylabel('MSE')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nInterpretación: el gráfico derecho muestra por qué el algoritmo 'busca' un mínimo:")
print("el error (MSE) es una función convexa de los parámetros, y el descenso de gradiente")
print("(o la fórmula cerrada de mínimos cuadrados) encuentra exactamente el punto más bajo de esa curva.")

### 0.7 ¿Cómo aplica PyCaret todo esto automáticamente?

Cuando ejecutemos `setup()` y `compare_models()` más adelante, PyCaret realizará —para cada uno de los ~20 algoritmos— el mismo proceso conceptual que acabamos de ilustrar a mano:

1. Ajustar los parámetros internos del modelo minimizando una función de error sobre los datos de entrenamiento (ya sea por fórmula cerrada, descenso de gradiente, o construcción recursiva de árboles).
2. Evaluar el modelo mediante validación cruzada de 10 folds.
3. Reportar las métricas (MAE, MSE, RMSE, R², RMSLE, MAPE) promediadas entre folds.
4. Ordenar los modelos según la métrica elegida (`RMSE` en nuestro caso).

Con esta base conceptual, pasamos ahora a la implementación completa sobre el dataset real de rendimiento académico.

---


## 1️⃣ Instalación de librerías

Instalamos PyCaret (versión completa, incluye módulos de interpretabilidad como SHAP), la librería oficial de Kaggle para descargar el dataset, y `scipy` para las pruebas estadísticas de validación que usaremos más adelante.

> ⚠️ **Nota:** Después de instalar, Colab puede pedirte **reiniciar el entorno de ejecución** (Runtime → Restart runtime). Esto es normal por conflictos de versiones de dependencias. Hazlo una sola vez y luego continúa ejecutando desde la siguiente celda.


In [ ]:
!pip install pycaret[full] kaggle scipy -q
print("✅ Instalación completada. Si Colab te pide reiniciar el entorno, hazlo y continúa desde la siguiente celda.")

## 2️⃣ Configuración de la API de Kaggle

Para descargar el dataset directamente desde Kaggle necesitas tu archivo de credenciales `kaggle.json`:

1. Entra a tu cuenta de Kaggle → **Settings** → sección **API** → botón **"Create New Token"**.
2. Se descargará un archivo llamado `kaggle.json`.
3. Ejecuta la celda siguiente y súbelo cuando se te solicite.


In [ ]:
import os

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except Exception:
    colab_files = None
    IN_COLAB = False

if IN_COLAB:
    print("📁 Sube tu archivo kaggle.json...")
    uploaded = colab_files.upload()

    os.makedirs('/root/.kaggle', exist_ok=True)
    if os.path.exists('kaggle.json'):
        !cp kaggle.json /root/.kaggle/
        !chmod 600 /root/.kaggle/kaggle.json
    print("✅ Credenciales de Kaggle configuradas correctamente.")
else:
    print("🧪 Ejecución local detectada. Se usará el dataset ya disponible en dataset/StudentsPerformance.csv.")


## 3️⃣ Descarga del dataset desde Kaggle

Descargamos y descomprimimos el dataset **Students Performance in Exams**.


In [ ]:
import os
import subprocess

if os.path.exists('dataset/StudentsPerformance.csv'):
    print('✅ Dataset ya disponible localmente en dataset/StudentsPerformance.csv')
else:
    try:
        subprocess.run(['kaggle', 'datasets', 'download', '-d', 'spscientist/students-performance-in-exams', '--force'], check=True)
        subprocess.run(['unzip', '-o', 'students-performance-in-exams.zip', '-d', 'dataset/'], check=True)
    except Exception as e:
        print('No se pudo descargar el dataset automáticamente:', e)
        raise

print('Archivos disponibles:')
print(os.listdir('dataset'))


## 4️⃣ Carga de datos y validación del esquema

Antes de cualquier análisis, **validamos la calidad e integridad del dataset**. Esta etapa es importante mencionarla en el capítulo de metodología de la tesis, ya que demuestra rigor en el manejo de los datos antes del modelado.

Validaciones que realizamos:
- Las columnas esperadas existen y con los nombres correctos.
- Los tipos de dato son los esperados (numéricos vs. categóricos).
- Los puntajes están en el rango válido (0-100).
- No hay valores nulos.
- No hay filas completamente duplicadas.
- Las categorías de las variables categóricas son las esperadas (sin valores inconsistentes, ej. "Male" vs "male").


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('dataset/StudentsPerformance.csv')

print(f"Dimensiones del dataset: {df.shape[0]} filas x {df.shape[1]} columnas\n")
df.head(10)

In [ ]:
# ---------------------------------------------------------------
# FUNCIÓN DE VALIDACIÓN DE DATOS (Data Validation)
# ---------------------------------------------------------------
def validar_dataset(data: pd.DataFrame) -> None:
    \"\"\"Ejecuta validaciones de calidad sobre el dataset y reporta resultados.\"\"\"

    print("="*60)
    print("REPORTE DE VALIDACIÓN DE DATOS")
    print("="*60)

    # 1. Columnas esperadas
    columnas_esperadas = ['gender', 'race/ethnicity', 'parental level of education',
                           'lunch', 'test preparation course',
                           'math score', 'reading score', 'writing score']
    faltantes = set(columnas_esperadas) - set(data.columns)
    print(f"[1] Columnas esperadas presentes: {'✅ SI' if not faltantes else f'❌ NO — faltan: {faltantes}'}")

    # 2. Tipos de dato
    columnas_numericas = ['math score', 'reading score', 'writing score']
    tipos_ok = all(pd.api.types.is_numeric_dtype(data[c]) for c in columnas_numericas)
    print(f"[2] Tipos de dato numéricos correctos: {'✅ SI' if tipos_ok else '❌ NO'}")

    # 3. Rango válido de puntajes (0-100)
    fuera_de_rango = {}
    for c in columnas_numericas:
        n_invalidos = ((data[c] < 0) | (data[c] > 100)).sum()
        if n_invalidos > 0:
            fuera_de_rango[c] = n_invalidos
    print(f"[3] Puntajes dentro del rango 0-100: {'✅ SI' if not fuera_de_rango else f'❌ NO — {fuera_de_rango}'}")

    # 4. Valores nulos
    nulos = data.isnull().sum().sum()
    print(f"[4] Sin valores nulos: {'✅ SI' if nulos == 0 else f'❌ NO — {nulos} valores nulos'}")

    # 5. Filas duplicadas
    duplicados = data.duplicated().sum()
    print(f"[5] Sin filas duplicadas: {'✅ SI' if duplicados == 0 else f'⚠️ {duplicados} duplicados encontrados'}")

    # 6. Categorías consistentes
    print("[6] Categorías únicas por variable categórica:")
    for c in ['gender', 'race/ethnicity', 'parental level of education', 'lunch', 'test preparation course']:
        print(f"     - {c}: {sorted(data[c].unique())}")

    print("="*60)
    print("Validación finalizada.\n")

validar_dataset(df)

In [ ]:
# Estadísticas descriptivas de las variables numéricas
df.describe()

## 5️⃣ Análisis Exploratorio de Datos (EDA)

Analizamos en profundidad la distribución de la variable objetivo, su relación con las demás variables numéricas, el efecto de las variables categóricas, y la presencia de valores atípicos.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
output_dir = Path('plots')
output_dir.mkdir(exist_ok=True)

# --- Gráfico 1: Distribuciones de los tres puntajes ---
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

sns.histplot(df['math score'], kde=True, ax=axes[0], color='steelblue')
axes[0].axvline(df['math score'].mean(), color='red', linestyle='--', label=f"Media: {df['math score'].mean():.1f}")
axes[0].set_title('Distribución - Math Score')
axes[0].legend()

sns.histplot(df['reading score'], kde=True, ax=axes[1], color='seagreen')
axes[1].axvline(df['reading score'].mean(), color='red', linestyle='--', label=f"Media: {df['reading score'].mean():.1f}")
axes[1].set_title('Distribución - Reading Score')
axes[1].legend()

sns.histplot(df['writing score'], kde=True, ax=axes[2], color='indianred')
axes[2].axvline(df['writing score'].mean(), color='red', linestyle='--', label=f"Media: {df['writing score'].mean():.1f}")
axes[2].set_title('Distribución - Writing Score')
axes[2].legend()

plt.tight_layout()
plt.savefig(output_dir / 'distribucion_puntajes.png', dpi=200)
plt.show()

print('Interpretación: las tres variables presentan una distribución aproximadamente normal,')
print('ligeramente sesgada, sin valores extremos evidentes en la media.')


In [ ]:
plt.figure(figsize=(6, 5))
corr = df[['math score', 'reading score', 'writing score']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', vmin=0, vmax=1)
plt.title('Correlación entre puntajes')
plt.tight_layout()
plt.savefig('plots/matriz_correlacion.png', dpi=200)
plt.show()

print('Interpretación esperada: reading score y writing score suelen estar altamente')
print('correlacionados entre sí (>0.9) y moderadamente correlacionados con math score.')
print('Esto es relevante para justificar su inclusión como predictores.')


In [ ]:
sns.pairplot(df[['math score', 'reading score', 'writing score']], diag_kind='kde', plot_kws={'alpha': 0.5, 'color': 'steelblue'})
plt.suptitle('Relación conjunta entre puntajes', y=1.02)
plt.savefig('plots/pairplot_puntajes.png', dpi=200)
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
categoricas = ['gender', 'race/ethnicity', 'parental level of education', 'lunch', 'test preparation course']

for ax, col in zip(axes.flatten(), categoricas):
    sns.countplot(data=df, x=col, ax=ax, palette='viridis', order=df[col].value_counts().index)
    ax.set_title(f'Distribución de {col}')
    ax.tick_params(axis='x', rotation=30)

axes.flatten()[-1].axis('off')
plt.tight_layout()
plt.savefig('plots/conteo_categoricas.png', dpi=200)
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.boxplot(data=df, x='gender', y='math score', ax=axes[0,0], palette='pastel')
axes[0,0].set_title('Math Score por Género')

sns.boxplot(data=df, x='lunch', y='math score', ax=axes[0,1], palette='pastel')
axes[0,1].set_title('Math Score por tipo de Almuerzo (proxy socioeconómico)')

sns.boxplot(data=df, x='test preparation course', y='math score', ax=axes[1,0], palette='pastel')
axes[1,0].set_title('Math Score por Curso de Preparación')

order = df.groupby('parental level of education')['math score'].mean().sort_values().index
sns.boxplot(data=df, x='parental level of education', y='math score', order=order, ax=axes[1,1], palette='pastel')
axes[1,1].set_title('Math Score por Nivel Educativo de los Padres')
axes[1,1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('plots/boxplots_variables_categoricas.png', dpi=200)
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
sns.violinplot(data=df, x='race/ethnicity', y='math score', palette='Set3')
plt.title('Distribución de Math Score por grupo étnico')
plt.tight_layout()
plt.savefig('plots/violin_race_ethnicity.png', dpi=200)
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df[['math score', 'reading score', 'writing score']], palette='Set2')
plt.title('Detección de valores atípicos (outliers)')
plt.tight_layout()
plt.savefig('plots/outliers_boxplot.png', dpi=200)
plt.show()

print('Conteo de valores atípicos por variable (método IQR):')
for col in ['math score', 'reading score', 'writing score']:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lim_inf, lim_sup = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    outliers = ((df[col] < lim_inf) | (df[col] > lim_sup)).sum()
    print(f'  - {col}: {outliers} outliers (límites: [{lim_inf:.1f}, {lim_sup:.1f}])')


## 6️⃣ Configuración del experimento en PyCaret

Definimos `math score` como variable objetivo. PyCaret se encargará automáticamente de:
- Codificar las variables categóricas (One-Hot Encoding).
- Normalizar las variables numéricas.
- Detectar y tratar valores atípicos (`remove_outliers=True`).
- Dividir los datos en entrenamiento (80%) y prueba (20%).
- Configurar validación cruzada (10-fold) para todos los modelos.

Todos los parámetros usados quedan documentados aquí para el capítulo de metodología.


In [ ]:
from pycaret.regression import *

s = setup(
    data=df,
    target='math score',
    train_size=0.8,
    normalize=True,
    normalize_method='zscore',
    transformation=False,
    remove_outliers=True,
    outliers_threshold=0.05,
    categorical_features=['gender', 'race/ethnicity', 'parental level of education',
                           'lunch', 'test preparation course'],
    numeric_features=['reading score', 'writing score'],
    fold=10,
    session_id=123,
    verbose=True
)

In [ ]:
# Resumen del experimento configurado (útil para documentar en la tesis)
setup_df = pull()
setup_df

## 7️⃣ Comparación automática de modelos

`compare_models()` entrena y evalúa (mediante validación cruzada de 10 folds) todos los algoritmos de regresión disponibles en PyCaret: Regresión Lineal, Ridge, Lasso, Elastic Net, Random Forest, Extra Trees, Gradient Boosting, XGBoost, LightGBM, CatBoost, Support Vector Regression, KNN, Árboles de Decisión, entre otros.

Ordenamos por **RMSE** (Root Mean Squared Error), una métrica estándar y fácilmente interpretable en la misma escala que la variable objetivo (puntos de puntaje).


In [ ]:
best_models = compare_models(sort='RMSE', n_select=3)
print("\n🏆 Top 3 modelos seleccionados:")
for i, m in enumerate(best_models, 1):
    print(f"{i}. {m}")

In [ ]:
# Tabla comparativa completa de métricas (para incluir en el capítulo de resultados)
results_df = pull()
results_df

In [ ]:
results_df = pull()
results_df


In [ ]:
# --- Gráfico 9: Comparación visual de los modelos por R² ---
plt.figure(figsize=(10, 6))
sns.barplot(data=results_df.reset_index(), x='R2', y='Model', palette='crest')
plt.title('Comparación de modelos por R² (mayor es mejor)')
plt.xlabel('R²')
plt.tight_layout()
plt.show()

## 8️⃣ Optimización de hiperparámetros (Hyperparameter Tuning)

Tomamos el mejor modelo de la comparación y optimizamos sus hiperparámetros mediante búsqueda aleatoria con validación cruzada, buscando minimizar el RMSE.


In [ ]:
best_model = best_models[0]

# Guardamos las métricas del modelo ANTES del tuning para poder comparar después
metricas_antes = predict_model(best_model)
metricas_antes_df = pull()

tuned_model = tune_model(
    best_model,
    optimize='RMSE',
    n_iter=50,
    fold=10
)

print("\n📊 Hiperparámetros óptimos encontrados:")
print(tuned_model.get_params())

In [ ]:
# --- Gráfico 10: Comparación ANTES vs DESPUÉS del tuning ---
metricas_despues = predict_model(tuned_model)
metricas_despues_df = pull()

comparacion = pd.DataFrame({
    'Antes del tuning': metricas_antes_df.iloc[0][['MAE','MSE','RMSE','R2']],
    'Después del tuning': metricas_despues_df.iloc[0][['MAE','MSE','RMSE','R2']]
})
print(comparacion)

comparacion.T[['MAE','RMSE']].plot(kind='bar', figsize=(8,5), colormap='Set2')
plt.title('Comparación de métricas de error antes/después del tuning')
plt.ylabel('Valor de la métrica')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('plots/antes_despues_tuning.png', dpi=200)
plt.show()


import pandas as pd

# --- Gráfico 8: Comparación visual de los modelos por RMSE ---
plt.figure(figsize=(10, 6))
sns.barplot(data=results_df.reset_index(), x='RMSE', y='Model', palette='mako')
plt.title('Comparación de modelos por RMSE (menor es mejor)')
plt.xlabel('RMSE')
plt.tight_layout()
plt.savefig('plots/comparacion_rmse.png', dpi=200)
plt.show()

# --- Gráfico 9: Comparación visual de los modelos por R² ---
plt.figure(figsize=(10, 6))
sns.barplot(data=results_df.reset_index(), x='R2', y='Model', palette='crest')
plt.title('Comparación de modelos por R² (mayor es mejor)')
plt.xlabel('R²')
plt.tight_layout()
plt.savefig('plots/comparacion_r2.png', dpi=200)
plt.show()


In [ ]:
# --- Validación 1: Estabilidad entre folds ---
cv_results = pull()  # resultados del último tune_model / create_model con CV

# Volvemos a extraer el detalle por fold ejecutando create_model sobre el modelo ya tuneado
detalle_folds = pull()
print("Si la tabla no muestra el detalle por fold, se puede regenerar con create_model():")

fold_model = create_model(tuned_model, fold=10)
fold_results = pull()
fold_results

In [ ]:
# --- Gráfico 11: Estabilidad del RMSE entre folds ---
folds_metricas = fold_results.iloc[:-2]  # excluye filas de Mean y SD
plt.figure(figsize=(10, 5))
plt.plot(folds_metricas.index.astype(str), folds_metricas['RMSE'], marker='o', color='darkorange')
plt.axhline(fold_results.loc['Mean', 'RMSE'], color='gray', linestyle='--',
            label=f"Media RMSE: {fold_results.loc['Mean','RMSE']:.2f}")
plt.title('Estabilidad del RMSE entre los 10 folds de validación cruzada')
plt.xlabel('Fold')
plt.ylabel('RMSE')
plt.legend()
plt.tight_layout()
plt.show()

std_rmse = fold_results.loc['SD', 'RMSE']
print(f"Desviación estándar del RMSE entre folds: {std_rmse:.3f}")
print("Una desviación baja (relativa a la media) indica un modelo ESTABLE y confiable.")

In [ ]:
# --- Gráfico 12: Curva de aprendizaje (Learning Curve) ---
plot_model(tuned_model, plot='learning')

In [ ]:
# --- Gráfico 13: Curva de validación (Validation Curve) ---
plot_model(tuned_model, plot='vc')

## 🔟 Evaluación visual adicional del modelo

PyCaret ofrece gráficos de diagnóstico estándar para modelos de regresión:
- **Residuals plot:** para verificar homocedasticidad (varianza constante de errores).
- **Prediction error plot:** compara valores reales vs. predichos.
- **Feature importance:** variables con mayor peso en la predicción.
- **Cook's distance:** identifica observaciones influyentes/atípicas para el modelo.


In [ ]:
plot_model(tuned_model, plot='residuals')

In [ ]:
plot_model(tuned_model, plot='error')

In [ ]:
plot_model(tuned_model, plot='feature')

In [ ]:
try:
    plot_model(tuned_model, plot='cooks')
except Exception as e:
    print("Este gráfico no está disponible para el tipo de modelo seleccionado.")
    print(e)

In [ ]:
# Panel resumen con todas las gráficas de diagnóstico en una sola vista
evaluate_model(tuned_model)

### Gráficas adicionales según el tipo de modelo

- Si el mejor modelo es **lineal** (Regresión Lineal, Ridge, Lasso, Elastic Net), el gráfico de **coeficientes** muestra el peso e importancia de cada variable, replicando visualmente los conceptos de la sección 0.2.
- Si el mejor modelo es un **árbol individual** (Decision Tree), se puede visualizar su estructura completa.
- El gráfico de **manifold** proyecta los datos en 2D (usando t-SNE/manifold learning) para observar visualmente si existen agrupaciones naturales relacionadas con el error de predicción.
- El gráfico de **parámetros** lista todos los hiperparámetros del modelo final, útil para reportarlos textualmente en la tesis.


In [ ]:
# --- Gráfico: Coeficientes (solo aplica a modelos lineales) ---
try:
    plot_model(tuned_model, plot='parameter')
except Exception as e:
    print("No se pudo generar el gráfico de parámetros para este modelo.")
    print(e)

In [ ]:
# --- Gráfico: estructura del árbol (solo aplica a modelos de árbol individual, ej. 'dt') ---
try:
    plot_model(tuned_model, plot='tree')
except Exception as e:
    print("Este modelo no es un árbol individual, por lo que no aplica este gráfico.")
    print("(Random Forest, XGBoost, etc. están compuestos por muchos árboles, no uno solo).")

In [ ]:
# --- Gráfico: proyección Manifold (agrupamiento visual en 2D) ---
try:
    plot_model(tuned_model, plot='manifold')
except Exception as e:
    print("Este gráfico no está disponible para el tipo de modelo/datos actual.")
    print(e)

## 1️⃣1️⃣ Interpretabilidad del modelo (SHAP)

Los valores **SHAP (SHapley Additive exPlanations)** permiten explicar, de forma cuantitativa, cómo cada variable contribuye a la predicción individual de cada estudiante. Esta sección es especialmente valiosa para el capítulo de **discusión de resultados** de una tesis, ya que permite ir más allá del "qué predice" hacia el "por qué lo predice".

> ⚠️ `interpret_model()` funciona nativamente con modelos basados en árboles (Random Forest, XGBoost, LightGBM, CatBoost, Extra Trees). Si el mejor modelo no es de este tipo, se debe reentrenar un modelo de árbol para esta sección (ver celda de respaldo).


In [ ]:
try:
    interpret_model(tuned_model, plot='summary')
except Exception as e:
    print("El modelo seleccionado no soporta SHAP nativamente.")
    print("Entrenando un modelo de respaldo basado en árboles (Random Forest) para interpretabilidad...\n")
    rf_model = create_model('rf')
    interpret_model(rf_model, plot='summary')

In [ ]:
# Interpretabilidad para dos observaciones distintas del set de prueba
for obs in [0, 5]:
    print(f"\n--- Explicación para la observación #{obs} ---")
    try:
        interpret_model(tuned_model, plot='reason', observation=obs)
    except Exception as e:
        interpret_model(rf_model, plot='reason', observation=obs)

In [ ]:
# Gráfico de dependencia SHAP: relación entre una variable y su impacto en la predicción
try:
    interpret_model(tuned_model, plot='correlation')
except Exception as e:
    interpret_model(rf_model, plot='correlation')

## 1️⃣2️⃣ Ensamblado de modelos (Ensemble Learning)

Como estrategia adicional —muy valorada en trabajos de tesis por demostrar dominio metodológico— combinamos los 3 mejores modelos mediante:

- **Blending:** promedio (ponderado o simple) de las predicciones de varios modelos.
- **Stacking:** un meta-modelo aprende a combinar las predicciones de los modelos base.

Comparamos su desempeño contra el mejor modelo individual optimizado.


In [ ]:
blended_model = blend_models(estimator_list=best_models, optimize='RMSE')

In [ ]:
stacked_model = stack_models(estimator_list=best_models, optimize='RMSE')

In [ ]:
# Comparación final entre: modelo tuneado, blending y stacking
resumen_comparacion = []

for nombre, modelo in [('Modelo Tuneado', tuned_model),
                        ('Blending', blended_model),
                        ('Stacking', stacked_model)]:
    predict_model(modelo)
    metricas = pull()
    metricas.insert(0, 'Estrategia', nombre)
    resumen_comparacion.append(metricas)

tabla_final = pd.concat(resumen_comparacion, ignore_index=True)
tabla_final[['Estrategia', 'MAE', 'MSE', 'RMSE', 'R2', 'RMSLE', 'MAPE']]

In [ ]:
# --- Gráfico 14: Comparación visual de las 3 estrategias ---
tabla_plot = tabla_final.set_index('Estrategia')[['RMSE', 'MAE']]
tabla_plot.plot(kind='bar', figsize=(9, 5), colormap='viridis')
plt.title('Comparación de estrategias: Tuneado vs Blending vs Stacking')
plt.ylabel('Valor de la métrica')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 1️⃣3️⃣ Validación final sobre el conjunto de prueba (hold-out)

Realizamos una validación estadística más rigurosa sobre las predicciones del modelo seleccionado en el conjunto de prueba (datos que el modelo nunca vio durante el entrenamiento ni el tuning):

- **Gráfico de dispersión real vs. predicho:** idealmente los puntos deben alinearse sobre la diagonal.
- **Histograma de residuos:** deben distribuirse aproximadamente de forma normal y centrados en cero.
- **Prueba de normalidad de residuos (Shapiro-Wilk):** valida un supuesto estadístico clásico de los modelos de regresión.
- **Umbral de aceptación:** definimos un criterio explícito (ej. R² ≥ 0.70) para decidir si el modelo es apto para uso práctico.

> ✏️ **Ajusta `modelo_seleccionado` según cuál estrategia (tuneado, blending o stacking) obtuvo el mejor resultado en la tabla anterior.**


In [ ]:
from scipy import stats

# Selecciona aquí el modelo con mejor desempeño observado en el paso anterior
modelo_seleccionado = tuned_model  # Cambia a blended_model o stacked_model si corresponde

pred_holdout = predict_model(modelo_seleccionado)
metricas_holdout = pull()
print("Métricas finales sobre el conjunto de prueba (hold-out):")
print(metricas_holdout)

y_real = pred_holdout['math score']
y_pred = pred_holdout['prediction_label']
residuos = y_real - y_pred

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_real, y_pred, alpha=0.5, color='teal')
lims = [min(y_real.min(), y_pred.min()), max(y_real.max(), y_pred.max())]
plt.plot(lims, lims, 'r--', label='Predicción perfecta')
plt.xlabel('Valor real (math score)')
plt.ylabel('Valor predicho')
plt.title('Validación: Valores reales vs. predichos (hold-out)')
plt.legend()
plt.tight_layout()
plt.savefig('plots/real_vs_predicho.png', dpi=200)
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(residuos, kde=True, color='purple')
plt.axvline(0, color='red', linestyle='--')
plt.title('Distribución de los residuos (error = real - predicho)')
plt.xlabel('Residuo')
plt.tight_layout()
plt.savefig('plots/hist_residuos.png', dpi=200)
plt.show()

stat, p_valor = stats.shapiro(residuos)
print(f'Prueba de Shapiro-Wilk: estadístico={stat:.4f}, p-valor={p_valor:.4f}')
if p_valor > 0.05:
    print('✅ No se rechaza la hipótesis de normalidad de los residuos (p > 0.05).')
else:
    print('⚠️ Se rechaza la hipótesis de normalidad de los residuos (p <= 0.05).')


In [ ]:
# --- Criterio explícito de validación / aceptación del modelo ---
UMBRAL_R2_MINIMO = 0.70
UMBRAL_RMSE_MAXIMO = 8.0

r2_obtenido = metricas_holdout.iloc[0]['R2']
rmse_obtenido = metricas_holdout.iloc[0]['RMSE']

print("="*60)
print("CRITERIO DE ACEPTACIÓN DEL MODELO")
print("="*60)
print(f"R² obtenido: {r2_obtenido:.3f}  (umbral mínimo: {UMBRAL_R2_MINIMO})")
print(f"RMSE obtenido: {rmse_obtenido:.3f}  (umbral máximo: {UMBRAL_RMSE_MAXIMO})")

if r2_obtenido >= UMBRAL_R2_MINIMO and rmse_obtenido <= UMBRAL_RMSE_MAXIMO:
    print("\n✅ El modelo CUMPLE los criterios de aceptación y puede finalizarse para su uso.")
else:
    print("\n⚠️ El modelo NO cumple los criterios definidos. Se recomienda probar otros algoritmos,")
    print("   más feature engineering, o ajustar los umbrales según el contexto del proyecto.")
print("="*60)

## 1️⃣4️⃣ Finalización y guardado del modelo

Una vez validado, **finalizamos** el modelo: esto significa reentrenarlo utilizando el 100% de los datos disponibles (train + test), maximizando el aprendizaje antes de su despliegue. Luego lo guardamos junto con todo su pipeline de preprocesamiento.


In [ ]:
final_model = finalize_model(modelo_seleccionado)
print("✅ Modelo finalizado (entrenado con el 100% de los datos):")
print(final_model)

In [ ]:
save_model(final_model, 'modelo_rendimiento_academico_final')

from google.colab import files
files.download('modelo_rendimiento_academico_final.pkl')
print("✅ Modelo guardado y descargado como 'modelo_rendimiento_academico_final.pkl'")

## 1️⃣5️⃣ Ejemplos de uso del modelo

Presentamos tres formas típicas de utilizar el modelo ya entrenado, simulando un escenario de despliegue real:

1. **Predicción individual:** un solo estudiante nuevo.
2. **Predicción por lote (batch):** varios estudiantes a la vez, como si vinieran de un archivo CSV subido por un usuario.
3. **Carga del modelo persistido:** demostramos que el archivo `.pkl` guardado puede cargarse de nuevo (por ejemplo, en otra sesión o en un sistema en producción) y usarse sin re-entrenar nada.


In [ ]:
# --- Ejemplo de uso 1: Predicción individual ---
def predecir_estudiante(modelo, gender, race_ethnicity, parental_education,
                         lunch, test_prep, reading_score, writing_score):
    \"\"\"Función reutilizable para predecir el math score de un solo estudiante.\"\"\"
    entrada = pd.DataFrame([{
        'gender': gender,
        'race/ethnicity': race_ethnicity,
        'parental level of education': parental_education,
        'lunch': lunch,
        'test preparation course': test_prep,
        'reading score': reading_score,
        'writing score': writing_score
    }])
    resultado = predict_model(modelo, data=entrada)
    return resultado['prediction_label'].iloc[0]

prediccion_1 = predecir_estudiante(
    final_model,
    gender='female',
    race_ethnicity='group B',
    parental_education="bachelor's degree",
    lunch='standard',
    test_prep='completed',
    reading_score=78,
    writing_score=80
)

print(f"📌 Predicción para el estudiante de ejemplo: math score ≈ {prediccion_1:.1f} puntos")

In [ ]:
# --- Ejemplo de uso 2: Predicción por lote (batch) ---
nuevos_estudiantes = pd.DataFrame({
    'gender': ['female', 'male', 'male', 'female'],
    'race/ethnicity': ['group B', 'group C', 'group A', 'group D'],
    'parental level of education': ["bachelor's degree", 'some college', 'high school', "master's degree"],
    'lunch': ['standard', 'free/reduced', 'standard', 'standard'],
    'test preparation course': ['completed', 'none', 'completed', 'none'],
    'reading score': [78, 60, 88, 92],
    'writing score': [80, 58, 85, 95]
})

predicciones_batch = predict_model(final_model, data=nuevos_estudiantes)
predicciones_batch[['gender', 'lunch', 'test preparation course',
                     'reading score', 'writing score', 'prediction_label']]

In [ ]:
# --- Gráfico 17: Visualización de las predicciones por lote ---
plt.figure(figsize=(8, 5))
sns.barplot(x=predicciones_batch.index.astype(str), y='prediction_label',
            data=predicciones_batch, palette='crest')
plt.title('Predicciones de Math Score para el lote de nuevos estudiantes')
plt.xlabel('Estudiante (índice)')
plt.ylabel('Math Score predicho')
plt.tight_layout()
plt.show()

In [ ]:
# --- Ejemplo de uso 3: Cargar el modelo persistido y validar que funciona igual ---
modelo_cargado = load_model('modelo_rendimiento_academico_final')

prediccion_verificacion = predict_model(modelo_cargado, data=nuevos_estudiantes)
print("Verificación de persistencia del modelo:")
print(prediccion_verificacion[['prediction_label']].head())

# Comprobación automática: las predicciones deben coincidir con las originales
coincide = np.allclose(
    predicciones_batch['prediction_label'].values,
    prediccion_verificacion['prediction_label'].values
)
print(f"\n¿Las predicciones del modelo cargado coinciden con las originales? {'✅ SI' if coincide else '❌ NO'}")

## 1️⃣6️⃣ Conclusiones y recomendaciones para el documento de tesis

### Cómo reportar estos resultados en la tesis

1. **Capítulo de Metodología:** describir el dataset (fuente, tamaño, variables), las validaciones de calidad de datos aplicadas, el preprocesamiento (`setup()` de PyCaret con sus parámetros), y el protocolo de validación (10-fold CV, partición 80/20).
2. **Capítulo de Resultados:** incluir la tabla comparativa de `compare_models()` (todas las métricas), las gráficas de estabilidad entre folds, curva de aprendizaje, residuos, importancia de variables, y la comparación entre modelo tuneado vs. blending vs. stacking.
3. **Capítulo de Discusión:** apoyarse en los valores SHAP para argumentar *por qué* ciertas variables (ej. `writing score`, `lunch`, `test preparation course`) influyen en el rendimiento en matemáticas, conectando con literatura de Learning Analytics. Incluir también la prueba de normalidad de residuos y el criterio explícito de aceptación del modelo como evidencia de rigor estadístico.
4. **Limitaciones a mencionar:** el dataset es sintético/anonimizado (no georreferenciado a una institución real), por lo que las conclusiones deben enmarcarse como una prueba de concepto metodológica, no como una generalización poblacional.
5. **Trabajo futuro:** aplicar el mismo pipeline a datos institucionales reales, incorporar variables temporales (asistencia, historial de calificaciones) y comparar contra modelos de deep learning tabular.

### Resumen de lo construido en este notebook

- ✅ Validación de calidad de datos antes del modelado.
- ✅ EDA completo con 7 tipos de visualizaciones distintas.
- ✅ Comparación de ~20 algoritmos de regresión.
- ✅ Tuning de hiperparámetros con comparación antes/después.
- ✅ Validación de estabilidad entre folds, curva de aprendizaje y de validación.
- ✅ Interpretabilidad con SHAP (resumen, casos individuales y dependencia).
- ✅ Ensamblado (blending y stacking) con comparación visual.
- ✅ Validación estadística final sobre hold-out (Shapiro-Wilk) y criterio explícito de aceptación.
- ✅ Finalización, guardado y **3 ejemplos de uso** (individual, batch, y carga del modelo persistido).
